# TP 2 — Classification de chiffres manuscrits

**Objectifs**

- Travailler avec un dataset d'images (chiffres 8×8 du `digits` de scikit-learn).
- Construire un Pipeline `StandardScaler + classifieur`.
- Comparer 4 classifieurs : régression logistique, SVM RBF, arbre, forêt aléatoire.
- Analyser la matrice de confusion pour identifier les classes confondues.

**Durée indicative : 50 minutes.**

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

digits = load_digits()
X, y = digits.data, digits.target  # X shape (1797, 64), y in {0..9}
print(X.shape, y.shape)

(1797, 64) (1797,)


## Exercice 1 — Aperçu visuel

Affichez les 12 premières images (`digits.images` de shape `(N, 8, 8)`) avec leur label en titre, sur une grille 3×4.

In [2]:
# TODO


## Exercice 2 — Split + Pipeline

1. Faites un split train/test stratifié 80/20.
2. Construisez un Pipeline `[StandardScaler, LogisticRegression(max_iter=2000)]`.
3. Entraînez-le et mesurez l'accuracy sur le test.
4. Affichez le `classification_report`.

<details>
<summary>💡 Coup de pouce — Pipeline scikit-learn</summary>

**🎯 But :** chaîner preprocessing + classifieur dans un seul objet qui se comporte comme un classifieur unique. Évite les fuites de données et simplifie le code.

**Construire une Pipeline**

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2000)),
])
```

**Anatomie**

- Liste de tuples `(nom, estimateur)`.
- Les **noms** (`'scaler'`, `'clf'`) sont libres mais doivent être **uniques**. Ils seront utilisés en TP9 avec GridSearchCV.
- Tous les estimateurs sauf le dernier doivent être des **transformers** (avec `.transform()`). Le dernier peut être un classifieur, régresseur, ou un transformer si on l'utilise pour transformer.

**Pourquoi `max_iter=2000` ?**

Par défaut, `LogisticRegression` fait 100 itérations. Sur Digits (64 features, 10 classes), ça ne suffit pas toujours pour converger. `max_iter=2000` évite le warning « ConvergenceWarning » sans changer le résultat final.

**Utilisation**

L'API est **identique à un classifieur** :

```python
pipe.fit(X_train, y_train)          # fit le scaler PUIS le classifieur
y_pred = pipe.predict(X_test)        # transform PUIS predict
score  = pipe.score(X_test, y_test)
```

**L'astuce anti-leakage**

Une Pipeline garantit que `StandardScaler.fit()` n'utilise **que `X_train`**. Si vous standardisez manuellement avant de splitter, vous calculez la moyenne sur train+test mélangés → leakage subtil qui gonfle les scores. La Pipeline évite ça automatiquement.

</details>

In [3]:
# TODO


## Exercice 3 — Comparaison de 4 classifieurs

Construisez un dictionnaire de 4 Pipelines :

- `"logreg"` : `StandardScaler + LogisticRegression(max_iter=2000)`
- `"svm"`    : `StandardScaler + SVC(kernel="rbf", C=10, gamma="scale")`
- `"tree"`   : `DecisionTreeClassifier(max_depth=10, random_state=0)` (pas besoin de scaler)
- `"rf"`     : `RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1)`

Pour chacun, entraînez sur le train et affichez l'accuracy sur le test. Faites un graphique en barres comparant les 4 scores.

<details>
<summary>💡 Coup de pouce — comparer 4 classifieurs</summary>

**🎯 But :** organiser proprement le benchmark de plusieurs modèles sur le même dataset.

**Stocker les modèles dans un dict**

```python
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

models = {
    'logreg': Pipeline([('scaler', StandardScaler()),
                        ('clf', LogisticRegression(max_iter=2000))]),
    'svm':    Pipeline([('scaler', StandardScaler()),
                        ('clf', SVC(kernel='rbf', gamma='scale'))]),
    'knn':    Pipeline([('scaler', StandardScaler()),
                        ('clf', KNeighborsClassifier(n_neighbors=5))]),
    'rf':     RandomForestClassifier(n_estimators=100, random_state=0),
}
```

⚠️ Notez que **RandomForest n'a pas besoin de standardisation** (les arbres comparent par seuil, pas par distance). On peut donc l'instancier sans Pipeline.

**Boucler proprement**

```python
scores = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    scores[name] = model.score(X_test, y_test)
    print(f"{name:8s}: {scores[name]:.3f}")
```

`{name:8s}` aligne les noms sur 8 caractères → sortie lisible :
```
logreg  : 0.969
svm     : 0.989
knn     : 0.986
rf      : 0.964
```

**Visualisation**

```python
plt.bar(scores.keys(), scores.values())
plt.ylim(0.9, 1.0)        # zoome sur la zone intéressante
plt.ylabel('accuracy test')
plt.xticks(rotation=15)
```

**Ce que vous devriez observer sur Digits**
- SVM RBF souvent en tête (≈ 0.98-0.99)
- k-NN proche derrière (≈ 0.98)
- LogReg solide (≈ 0.96)
- Random Forest correct mais pas meilleur ici (≈ 0.96) — Digits a peu de bruit, l'avantage des forêts ne ressort pas.

</details>

In [4]:
# TODO


## Exercice 4 — Matrice de confusion du meilleur modèle

Affichez la matrice de confusion 10×10 du meilleur modèle (`ConfusionMatrixDisplay.from_estimator`).

Identifiez les paires de chiffres les plus confondues. Affichez 6 exemples mal classés avec en titre `(vrai, prédit)` pour visualiser.

<details>
<summary>💡 Coup de pouce — analyser les erreurs du meilleur modèle</summary>

**🎯 But :** ne pas se contenter du score global, mais comprendre **où** le modèle se trompe.

**Identifier les paires les plus confondues**

La matrice de confusion `cm` est de shape `(n_classes, n_classes)`. Les valeurs **hors diagonale** sont les erreurs :

```python
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)

import numpy as np
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)        # met la diagonale à 0 pour ignorer les bons cas
i, j = np.unravel_index(np.argmax(cm_off), cm.shape)
print(f"Paire la plus confondue : vrai={i} prédit={j} ({cm_off[i, j]} erreurs)")
```

Sur Digits, attendez-vous à voir des paires graphiquement proches : 1↔7, 3↔8, 4↔9, etc.

**Récupérer les exemples mal classés**

```python
wrong_idx = np.where(y_pred != y_test)[0]
print(f"{len(wrong_idx)} erreurs au total")
```

`np.where` retourne les **indices** où la condition est vraie. `[0]` car `where` renvoie un tuple (un élément par axe).

**Afficher les 6 premières erreurs**

```python
fig, axes = plt.subplots(1, 6, figsize=(12, 2))
for ax, idx in zip(axes, wrong_idx[:6]):
    img_8x8 = X_test[idx].reshape(8, 8)
    ax.imshow(img_8x8, cmap='gray_r')
    ax.set_title(f"vrai:{y_test[idx]} | pred:{y_pred[idx]}", fontsize=9)
    ax.axis('off')
```

⚠️ **Reshape 8×8** : les digits sklearn sont stockés en vecteurs aplatis de 64. Pour les afficher, il faut les remettre en image 8×8.

**Ce que ça révèle**

En regardant les images, vous verrez souvent que :
- Le modèle se trompe sur des digits **graphiquement ambigus** (un 4 qui ressemble vraiment à un 9).
- Certaines erreurs sont **excusables** : même un humain hésiterait.
- D'autres sont **incompréhensibles** : signe que le modèle est en limite de capacité.

C'est cette analyse qui guide les améliorations (data augmentation, modèle plus puissant, etc.).

</details>

In [5]:
# TODO
